<a href="https://colab.research.google.com/github/ryanp-cloud/ai-native-analytics/blob/main/01_revenue_reconciliation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install the libraries used for analytics development
%pip install -q duckdb pandas pytest

In [2]:
# Import the analytics libraries
import duckdb
import pandas as pd

# Create an in-memory analytical database
con = duckdb.connect()

# Verify that DuckDB is working
result = con.execute("SELECT 42 AS answer").df()

display(result)

,answer
0,42


In [3]:
# Create a sample billing dataset
billing_df = pd.DataFrame({
    "invoice_id": ["INV001", "INV002", "INV003", "INV004", "INV005"],
    "customer_id": ["C001", "C002", "C003", "C004", "C005"],
    "amount": [1200, 850, 2400, 1500, 900]
})

# Register the DataFrame as a DuckDB table
con.register("billing", billing_df)

# Preview the billing data
display(con.execute("SELECT * FROM billing").df())

,invoice_id,customer_id,amount
0,INV001,C001,1200
1,INV002,C002,850
2,INV003,C003,2400
3,INV004,C004,1500
4,INV005,C005,900


In [4]:
# Create a sample accounting dataset
accounting_df = pd.DataFrame({
    "invoice_id": ["INV001", "INV002", "INV003", "INV004", "INV006"],
    "customer_id": ["C001", "C002", "C003", "C004", "C006"],
    "amount": [1200, 800, 2400, 1500, 600]
})

# Register the DataFrame as a DuckDB table
con.register("accounting", accounting_df)

# Preview the accounting data
display(con.execute("SELECT * FROM accounting").df())

,invoice_id,customer_id,amount
0,INV001,C001,1200
1,INV002,C002,800
2,INV003,C003,2400
3,INV004,C004,1500
4,INV006,C006,600


In [5]:
# Compare total revenue and transaction counts
validation = con.execute("""
    SELECT
        'Billing' AS source,
        COUNT(*) AS transactions,
        SUM(amount) AS total_revenue
    FROM billing

    UNION ALL

    SELECT
        'Accounting' AS source,
        COUNT(*) AS transactions,
        SUM(amount) AS total_revenue
    FROM accounting
""").df()

display(validation)

,source,transactions,total_revenue
0,Billing,5,6850.0
1,Accounting,5,6500.0


In [6]:
# Join billing and accounting data by invoice ID
reconciliation = con.execute("""
    SELECT
        COALESCE(b.invoice_id, a.invoice_id) AS invoice_id,
        b.amount AS billing_amount,
        a.amount AS accounting_amount

    FROM billing AS b

    FULL OUTER JOIN accounting AS a
        ON b.invoice_id = a.invoice_id

    ORDER BY invoice_id
""").df()

# Display invoice-level results
display(reconciliation)

,invoice_id,billing_amount,accounting_amount
0,INV001,1200,1200
1,INV002,850,800
2,INV003,2400,2400
3,INV004,1500,1500
4,INV005,900,<NA>
5,INV006,<NA>,600


In [7]:
# Reconcile every invoice between Billing and Accounting
reconciliation = con.execute("""
    SELECT
        COALESCE(b.invoice_id, a.invoice_id) AS invoice_id,

        b.amount AS billing_amount,
        a.amount AS accounting_amount,

        -- Treat missing amounts as zero when calculating variance
        COALESCE(b.amount, 0) - COALESCE(a.amount, 0)
            AS variance,

        -- Classify reconciliation results
        CASE
            WHEN b.invoice_id IS NULL
                THEN 'MISSING_IN_BILLING'

            WHEN a.invoice_id IS NULL
                THEN 'MISSING_IN_ACCOUNTING'

            WHEN b.amount <> a.amount
                THEN 'AMOUNT_MISMATCH'

            ELSE 'MATCHED'
        END AS reconciliation_status

    FROM billing AS b
    FULL OUTER JOIN accounting AS a
        ON b.invoice_id = a.invoice_id

    ORDER BY invoice_id
""").df()

display(reconciliation)

,invoice_id,billing_amount,accounting_amount,variance,reconciliation_status
0,INV001,1200,1200,0,MATCHED
1,INV002,850,800,50,AMOUNT_MISMATCH
2,INV003,2400,2400,0,MATCHED
3,INV004,1500,1500,0,MATCHED
4,INV005,900,<NA>,900,MISSING_IN_ACCOUNTING
5,INV006,<NA>,600,-600,MISSING_IN_BILLING
